# FCIS 2026 Scene Classification — Colab GPU training

This notebook trains five models on a Colab GPU: four course models (including a from-scratch ViT transformer) plus a pretrained ResNet18 bonus model. It downloads the competition data using your Kaggle API credentials, compares models on the same stratified validation split, and creates test predictions.

Before running: set Colab runtime to **T4 GPU** (Runtime → Change runtime type), accept the Kaggle competition rules, and add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Colab Secrets panel. Enable notebook access to those secrets. The notebook clones this public GitHub repository and downloads the competition data directly from Kaggle; no dataset files or API keys are stored in GitHub.

In [ ]:
from google.colab import drive, files, userdata
from pathlib import Path
import os, zipfile, subprocess, sys, json, shutil

assert __import__('torch').cuda.is_available(), 'Enable a GPU runtime first: Runtime → Change runtime type → T4 GPU'
print(__import__('torch').cuda.get_device_name(0))
drive.mount('/content/drive')  # saves checkpoints and submissions beyond this Colab session
project = Path('/content/fcis-scene-classification')
if not project.exists(): subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dandan29376060887/fcis-scene-classification.git', str(project)], check=True)
os.chdir(project)
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
os.makedirs('/content/Data', exist_ok=True)
subprocess.run(['kaggle', 'competitions', 'download', '-c', 'fcis-2026-nn-top-view-scene-classification', '-p', '/content'], check=True)
archive = next(Path('/content').glob('fcis-2026-nn-top-view-scene-classification.zip'))
with zipfile.ZipFile(archive) as z: z.extractall('/content/Data')
print('Training images:', len(list(Path('/content/Data/data').glob('*.jpg'))), '| test images:', len(list(Path('/content/Data/test').glob('*.jpg'))))

In [ ]:
# Train all four scratch models (ViT satisfies the transformer requirement) and a pretrained bonus model.
models = ['simple_cnn', 'vgg_style', 'residual_cnn', 'se_cnn', 'vit_scratch', 'resnet18_transfer']
results = []
for name in models:
    cfg = json.loads(Path(f'configs/{name if name != "resnet18_transfer" else "residual_cnn"}.json').read_text())
    cfg['data'].update(root='/content/Data', batch_size=64, num_workers=2, image_size=224, seed=42, val_fraction=0.15)
    cfg['model']['name'] = name
    cfg['model']['pretrained'] = (name == 'resnet18_transfer')
    if name == 'vit_scratch':
        cfg['model']['extra'] = {'patch_size':16, 'depth':4, 'heads':4, 'mlp_ratio':4.0, 'image_size':224}
    cfg['train'].update(epochs=40, early_stopping_patience=8, lr=(3e-4 if name == 'vit_scratch' else 1e-3), out_dir=f'/content/drive/MyDrive/fcis_runs/{name}', seed=42)
    Path(f'/content/{name}.json').write_text(json.dumps(cfg, indent=2))
    print(f'\n===== Training {name} =====', flush=True)
    subprocess.run([sys.executable, 'scripts/train.py', '--config', f'/content/{name}.json'], check=True)
    history = json.loads(Path(cfg['train']['out_dir'], 'history.json').read_text())
    best = max(history, key=lambda row: row['val_acc'])
    results.append({'model': name, 'best_val_acc': best['val_acc'], 'best_epoch': best['epoch'], 'run_dir': cfg['train']['out_dir']})
    print(results[-1])
Path('/content/drive/MyDrive/fcis_runs/model_comparison.json').write_text(json.dumps(results, indent=2))
sorted(results, key=lambda x: x['best_val_acc'], reverse=True)

In [ ]:
# Create a Kaggle-format submission from the model with the best held-out validation accuracy.
winner = max(results, key=lambda x: x['best_val_acc'])
print('Selected on held-out validation:', winner['model'], winner['best_val_acc'])
winner_cfg = f"/content/{winner['model']}.json"
out_csv = '/content/drive/MyDrive/fcis_runs/submission.csv'
subprocess.run([sys.executable, 'scripts/predict.py', '--config', winner_cfg, '--checkpoint', f"{winner['run_dir']}/best.pt", '--data-root', '/content/Data', '--out', out_csv, '--tta'], check=True)
import pandas as pd
submission = pd.read_csv(out_csv)
assert list(submission.columns) == ['image_id', 'label'] and len(submission) == 1100
assert submission['label'].notna().all()
print('Saved:', out_csv, '| rows:', len(submission))
files.download(out_csv)